# DAN-MMGradCAM · COE 691 final project · main notebook

**Team:** Dwi Haryani (202527350), Ayesha Mateen (202524590), Nahida Zakir (202526270)

This notebook fetches the project code and archived experiment outputs from GitHub and runs the full analysis:

| Part | Content | Report |
|---|---|---|
| 1 | Data, patient-separated split and anchor replication | Table 1 |
| 2 | **Module 1**: intensity-aware, directional 1D–2D agreement against a mispairing null | Table 2, Figs. 2–3 |
| 3 | **Module 2**: prediction-preserving, branch-specific stability | Fig. 4 |
| 4 | **Module 3**: reliability-weighted fusion (3A) and waveform-conditioned fusion (3B) | Figs. 5–6 |
| 5 | Faithfulness and comparative evaluation | Tables 3–4, Fig. 7 |
| 6 | Pipeline and reproducibility audit (SHA-256 check of the archived outputs) | – |

**Code:** https://github.com/dwiharyani/dan-mmgradcam-anchor-replication, branch `coe691-final` (notebooks, plotting code, HPC pipeline scripts and archived outputs).
The anchor re-run is on branch `main`, the Grad-CAM alignment and Module 1 alignment on branch `module1-nahida`.

**Requirements:** Python ≥ 3.10, `git`, and `numpy pandas scipy scikit-learn matplotlib` (preinstalled in Google Colab). Run all cells from top to bottom (about 5 minutes).

## 0 · Fetch the code from GitHub

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/dwiharyani/dan-mmgradcam-anchor-replication.git'
BRANCH = 'coe691-final'
ROOT = Path('DAN-MMGradCAM').resolve()          # local folder for the cloned branch

def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    r.check_returncode()
    return r.stdout.strip()

if (ROOT / '.git').exists():
    git('-C', str(ROOT), 'pull', '--ff-only')
else:
    git('clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(ROOT))
commit = git('-C', str(ROOT), 'log', '-1', '--format=%h %ad %s', '--date=short')
print('branch', BRANCH, '| folder', ROOT.name)

## Setup

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(ROOT / 'src'))      # plotting functions from the cloned repository
ARCH = ROOT / 'data' / 'archived'          # archived experiment outputs from the cloned repository
TAB = ROOT / 'results' / 'tables'          # tables written by this notebook
FIG = ROOT / 'results' / 'figures'         # figures written by this notebook
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40); pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=RuntimeWarning)
CLASSES = ['AFib', 'GSVT', 'SB', 'SR']
def show(name, width=900):
    display(Image(filename=str(FIG / f'{name}.png'), width=width))
print('archived files:', sum(1 for p in ARCH.rglob('*') if p.is_file()))

# Part 1 · Data, patient-separated split and anchor replication

**Project:** DAN-MMGradCAM: reliability of paired 1D-signal and 2D-image Grad-CAM explanations for ECG arrhythmia classification (COE 691 final project).

This notebook covers the data and the classifiers that every explanation analysis depends on:

1. the Chapman–Shaoxing Lead-II cohort and the patient-separated split into train/validation/test,
2. the test accuracy of the replicated MM-GradCAM 1D and 2D CNNs, compared with the published anchor (Duranay et al., 2026),
3. the explanation cohort: test recordings classified correctly by both CNNs.

**Inputs** (archived HPC outputs in `data/archived/`): split manifests, test predictions written with the Grad-CAM maps, and the Module 2 record table (predictions on re-rendered images).
Training itself (`src/hpc_pipeline/01_data_and_training/`) needs the raw ECGs and a GPU and was run on the KFUPM HPC; see notebook 06.

### 1. Cohort and split

In [ ]:
meta = json.loads((ARCH / 'split/test_split_1054_metadata.json').read_text())
test = pd.read_csv(ARCH / 'split/test_split_1054.csv')
train = pd.read_csv(ARCH / 'split/train_split.csv')
N_TOTAL = 10588                      # final cohort, one recording per patient (frozen in build_frozen_split.py)
n_val = N_TOTAL - len(train) - len(test)
split = pd.DataFrame({'split': ['train', 'validation', 'test', 'total'],
                      'recordings': [len(train), n_val, len(test), N_TOTAL]})
assert len(train) == 8475 and n_val == 1059 and len(test) == 1054
assert set(train.cohort_index).isdisjoint(test.cohort_index), 'train and test overlap'
print('Overlap between train and test cohort indices:', len(set(train.cohort_index) & set(test.cohort_index)))
print('Test class counts (metadata):', meta['class_counts'])
split

In [ ]:
both = pd.concat([train.assign(split='train'), test.assign(split='test')], ignore_index=True)
counts = pd.crosstab(both.class_name, both.split)
counts.loc['total'] = counts.sum()
counts

Each recording is 10 s of Lead II sampled at 500 Hz (5000 samples). Because the cohort holds exactly one recording per patient,
a split by recording is also a split by patient. The z-score scaler of the 1D branch is fitted on the 9534 non-test recordings only.

### 2. Replication of the anchor CNNs

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_recall_fscore_support
p1 = pd.read_csv(ARCH / 'replication/1d_test_predictions.csv')
p2 = pd.read_csv(ARCH / 'replication/2d_test_predictions.csv')
assert len(p1) == len(p2) == 1054 and set(p1.cohort_index) == set(test.cohort_index)
ANCHOR = {'1D': 93.07, '2D': 97.44}          # accuracy reported by Duranay et al. (2026)
rows, per_class = [], []
for name, d in (('1D', p1), ('2D', p2)):
    y, yhat = d.true_class, d.predicted_class
    rows.append({'branch': name, 'anchor_acc_%': ANCHOR[name], 'replication_acc_%': 100 * accuracy_score(y, yhat),
                 'macro_F1_%': 100 * f1_score(y, yhat, average='macro'),
                 'UAR_%': 100 * recall_score(y, yhat, average='macro')})
    pr, rc, f1, sup = precision_recall_fscore_support(y, yhat, labels=CLASSES)
    for c, a, b, f, s in zip(CLASSES, pr, rc, f1, sup):
        per_class.append({'branch': name, 'class': c, 'precision_%': 100 * a, 'recall_%': 100 * b, 'F1_%': 100 * f, 'n': s})
rep = pd.DataFrame(rows); rep_cls = pd.DataFrame(per_class)
rep.to_csv(TAB / 'replication_accuracy.csv', index=False); rep_cls.to_csv(TAB / 'replication_per_class.csv', index=False)
display(rep.round(2))
rep_cls.round(1)

The 2D accuracy recomputed here (97.06 %) is from the predictions saved together with the Grad-CAM maps.
The training script's own evaluation of the selected checkpoint reported 96.68 %; that number comes from the HPC training log and is not recomputed here.

In [ ]:
import dan_figures as F
F.fig_confusion(p1, p2, FIG)
show('fig_replication_confusion', 650)

### 3. Explanation cohort

In [ ]:
rec = pd.read_csv(ARCH / 'module2/noise_stability_records.csv')
both_orig = (p1.set_index('cohort_index').correct & p2.set_index('cohort_index').eval('true_class == predicted_class'))
rec['correct_1d'] = rec.pred_1d == rec.class_name
rec['correct_2d_rerendered'] = rec.pred_2d == rec.class_name
agree_2d = (rec.set_index('cohort_index').pred_2d == p2.set_index('cohort_index').predicted_class).sum()
cohort = pd.DataFrame({
    'quantity': ['correct in both CNNs (original images)', '2D correct on re-rendered images',
                 '2D re-rendered prediction = original prediction', 'primary explanation cohort (both correct, re-rendered)'],
    'n': [int(both_orig.sum()), int(rec.correct_2d_rerendered.sum()), int(agree_2d),
          int((rec.correct_1d & rec.correct_2d_rerendered).sum())]})
cohort['% of 1054'] = 100 * cohort.n / 1054
cohort.to_csv(TAB / 'explanation_cohort.csv', index=False)
cohort.round(1)

In [ ]:
orig2 = p2.set_index('cohort_index').predicted_class
rr = rec.set_index('cohort_index')
same = rr.pred_2d == orig2
real_ok, rer_ok = orig2 == rr.class_name, rr.pred_2d == rr.class_name
diff = rr[~same]
print(f'2D: real images {100 * real_ok.mean():.2f} % correct, re-rendered images {100 * rer_ok.mean():.2f} % correct')
print(f'same prediction on real and re-rendered image: {int(same.sum())} of {len(rr)}')
print(f'correct on real but wrong on re-rendered: {int((real_ok & ~rer_ok).sum())} | the reverse: {int((~real_ok & rer_ok).sum())}')
print('class of the recordings with differing predictions:', diff.class_name.value_counts().to_dict())

In [ ]:
tab = rec.groupby('class_name').agg(n=('cohort_index', 'size'), acc_1d=('correct_1d', 'mean'),
                                    acc_2d_rerendered=('correct_2d_rerendered', 'mean'),
                                    both_correct=('correct_1d', lambda s: int((s & rec.loc[s.index, 'correct_2d_rerendered']).sum())))
tab.loc['All'] = [len(rec), rec.correct_1d.mean(), rec.correct_2d_rerendered.mean(), int((rec.correct_1d & rec.correct_2d_rerendered).sum())]
tab.round(3)

**Summary.** The replicated 1D CNN (94.78 %) exceeds the published 1D accuracy and the 2D CNN (97.06 %) is within 0.4 points of it.
Module 2 regenerates every image, including the unperturbed one, so all explanation analyses use the **939** test recordings that are correct in both branches under re-rendering.

# Part 2 · Module 1: intensity-aware and directional 1D–2D agreement

**Research question 1.** Do the 1D signal CNN and the 2D image CNN explain the same recording with the same temporal evidence, and is their agreement above chance?

**Method.** Both Grad-CAM maps are placed on one 600-column time grid (536 valid waveform columns).
The 2D map is projected onto the waveform with an explicit kernel that weights but never replaces its intensity,
$g_2(u)=\sum_y H(y,u)K(y,u)/\sum_y K(y,u)$, here an on-trace corridor of $\pm 6$ px (the projection used by Modules 2 and 3).
Each map is normalised to attribution mass, $p_u=g_1(u)/\sum g_1$, $q_u=g_2(u)/\sum g_2$, and compared with

* shared mass $O=\sum_u \min(p_u,q_u)$, weighted Jaccard $J_w=O/(2-O)$, total variation $1-O$,
* Spearman $\rho_s$, top-15 % Dice (chance level $k/T=0.15$),
* direction $r_u=(p_u-q_u)/(p_u+q_u)$, gated $|r_u|>1/3$ (one branch carries at least twice the mass) on the union of salient columns,
* normalised entropy of each map (1 = flat).

Every statistic is compared with a **within-class mispairing null**: the 1D map of one recording is paired with the 2D map of another recording of the same class.

**Module 1 branch analysis.** Section 5 reproduces the min–max agreement analysis of the Module 1 branch (`module1-nahida`) on all 1054 recordings: local agreement $A(u)=\sqrt{\hat g_1\hat g_2}$ and disagreement $D(u)=|\hat g_1-\hat g_2|$ of min–max-normalised profiles, class-wise statistics, the correctness subgroups, and the alignment-shift, projection-mask and temporal-validation sensitivity analyses.

In [ ]:
from scipy.stats import spearmanr, kruskal
EPS, TOP, GATE, N_NULL, N_BOOT, SEED = 1e-12, 0.15, 1/3, 20, 2000, 20261117
rng = np.random.default_rng(SEED)

def mass(x):
    x = np.clip(np.nan_to_num(np.asarray(x, float)), 0, None); s = x.sum()
    return None if s <= EPS else x / s

def topk(x, f):
    k = max(1, int(np.ceil(f * x.size))); s = np.zeros(x.size, bool)
    s[np.argsort(-x, kind='stable')[:k]] = True; return s

def entropy_norm(p):
    nz = p[p > 0]; return float(-(nz * np.log(nz)).sum() / np.log(p.size))

def agreement(p, q):
    """Module 1 agreement between two attribution-mass vectors on the same time grid."""
    O = float(np.minimum(p, q).sum()); a, b = topk(p, TOP), topk(q, TOP)
    r = (p - q) / (p + q + EPS); sal = a | b
    return {'O': O, 'Jw': O / (2 - O), 'TV': 1 - O, 'spearman': float(spearmanr(p, q).statistic),
            'dice15': 2 * (a & b).sum() / (a.sum() + b.sum()), 'chance_dice': a.sum() / p.size,
            'dir_1D_dominant': 100 * np.mean(sal & (r > GATE)), 'dir_2D_dominant': 100 * np.mean(sal & (r < -GATE)),
            'dir_balanced': 100 * np.mean(sal & (np.abs(r) <= GATE)),
            'entropy_1D': entropy_norm(p), 'entropy_2D': entropy_norm(q)}

def boot_ci(x, stat=np.median):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    b = [stat(rng.choice(x, x.size)) for _ in range(N_BOOT)]
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

### 1. Per-recording agreement

In [ ]:
z = np.load(ARCH / 'module2/noise_stability_maps.npz')
rec = pd.read_csv(ARCH / 'module2/noise_stability_records.csv')
assert (rec.cohort_index.to_numpy() == z['cohort_index']).all()
V = z['valid_600'].astype(bool)
rec['both_correct'] = (rec.pred_1d == rec.class_name) & (rec.pred_2d == rec.class_name)
rows, P, Q = [], {}, {}
for i, r in rec.iterrows():
    p, q = mass(z['base1'][i][V[i]]), mass(z['base2'][i][V[i]])
    if p is None or q is None:
        rows.append({'cohort_index': r.cohort_index, 'class_name': r.class_name, 'degenerate': True}); continue
    P[i], Q[i] = p, q
    rows.append({'cohort_index': r.cohort_index, 'class_name': r.class_name, 'degenerate': False,
                 'both_correct': r.both_correct, **agreement(p, q)})
per = pd.DataFrame(rows)
per.to_csv(TAB / 'module1_per_record.csv', index=False)
prim = per[(~per.degenerate) & (per.both_correct == True)]
print('valid columns per record:', V[0].sum(), '| degenerate maps:', int(per.degenerate.sum()), '| primary cohort:', len(prim))
prim.drop(columns=['degenerate', 'both_correct']).describe().T[['mean', '25%', '50%', '75%']].round(3)

### 2. Within-class mispairing null

In [ ]:
null = []
for c in CLASSES:
    idx = np.array([i for i in P if rec.class_name[i] == c and rec.both_correct[i]])
    for _ in range(N_NULL):
        for i, j in zip(idx, rng.permutation(idx)):
            if i != j:
                m = agreement(P[i], Q[j]); null.append({'class_name': c, 'O': m['O'], 'spearman': m['spearman'], 'dice15': m['dice15']})
null = pd.DataFrame(null); null.to_csv(TAB / 'module1_null_mispairing.csv', index=False)
print('mispaired 1D/2D pairs in the null:', len(null))

In [ ]:
m1 = []
for c in CLASSES + ['ALL']:
    g = prim if c == 'ALL' else prim[prim.class_name == c]
    nn = null if c == 'ALL' else null[null.class_name == c]
    row = {'class': c, 'n': len(g)}
    for k in ['O', 'Jw', 'spearman', 'dice15', 'entropy_1D', 'entropy_2D', 'dir_1D_dominant', 'dir_2D_dominant', 'dir_balanced']:
        row[k] = g[k].median()
    for k in ['O', 'spearman', 'dice15']:
        row[f'{k}_CI'] = '%.3f–%.3f' % boot_ci(g[k]); row[f'{k}_null'] = nn[k].median()
    thr = {cc: null[null.class_name == cc]['O'].quantile(.95) for cc in CLASSES}
    row['% records O > null95'] = 100 * np.mean([o > thr[cl] for o, cl in zip(g.O, g.class_name)])
    row['% records 2D-leaning'] = 100 * (g.dir_2D_dominant > g.dir_1D_dominant).mean()
    m1.append(row)
m1 = pd.DataFrame(m1); m1.to_csv(TAB / 'module1_agreement_by_class.csv', index=False)
kw = kruskal(*[prim[prim.class_name == c].O for c in CLASSES])
print(f'Kruskal-Wallis test of shared mass across classes: H = {kw.statistic:.1f}, p = {kw.pvalue:.2e}')
m1.set_index('class').T

**Reading.** The median shared mass of correctly paired recordings is only slightly higher than that of mispaired recordings of the same class,
rank correlation is close to zero, and Dice is close to its chance value 0.15. The 2D map is nearly flat (normalised entropy ≈ 0.97),
and among salient columns 2D-dominant time is more common than 1D-dominant time.

### 3. Case study and cohort figure

In [ ]:
import dan_figures as F
z_m = np.load(ARCH / 'module2/noise_stability_maps.npz')        # archived Grad-CAM maps (HPC run); base1/base2 = unperturbed ECG
mask209 = np.load(ARCH / 'module1/ecg209_trace_mask.npy')       # pixel mask of the ECG 209 waveform (drawing background only)
i209 = int(np.flatnonzero(z_m['cohort_index'] == 209)[0])
v = z_m['valid_600'][i209].astype(bool)
p209, q209 = F.mass(z_m['base1'][i209][v]), F.mass(z_m['base2'][i209][v])
print(f'ECG 209 ({z_m["filename"][i209]}): O recomputed from the maps = {np.minimum(p209, q209).sum():.3f}')
print(per[per.cohort_index == 209][['O', 'Jw', 'spearman', 'dice15', 'dir_1D_dominant', 'dir_2D_dominant']].round(3).to_string(index=False))
F.fig_case_module1(z_m, i209, mask209, FIG)
import dan_column_figures as C
C.fig_module1_case_column(z_m, i209, mask209, FIG)   # single-column version used in the report
show('fig_module1_case209_drawn', 900)

ECG 209 (AFib), drawn from the archived Grad-CAM maps of the unperturbed recording (`base1`, `base2` in `noise_stability_maps.npz`, written by
`src/hpc_pipeline/03_module2_stability/module2_common.py`): (a) aligned 1D intensity on the waveform; (b) 2D intensity received by the waveform inside the ±6 px corridor;
(c) shared intensity $\min(p,q)$; (d) direction among salient columns; (e) the two mass curves. Grad-CAM is not re-run here; panels (c)–(e) and $O$ are computed from the maps in this cell.

In [ ]:
import dan_figures as F
per_fig = per.copy(); per_fig['both_correct'] = per_fig.both_correct.fillna(False).astype(bool)
F.fig_module1_cohort(per_fig, null, FIG)
show('fig_module1_cohort', 700)

### 4. Comparison with the anchor-style binary-mask agreement and projection sensitivity

The Module 1 branch computes agreement after min–max scaling each projected map (Section 5).
Below, the same recordings are summarised with that measure next to the mass-based agreement of this notebook.

In [ ]:
mm = pd.read_csv(ARCH / 'module1/minmax_agreement_shift0.csv')
ok = mm[(mm.degenerate_1d.astype(str) == 'False') & (mm.degenerate_2d.astype(str) == 'False')]
base_cmp = pd.DataFrame({
    'method': ['min–max agreement (Module 1 branch)', 'intensity-aware mass (this work)'],
    'n': [len(ok), len(prim)],
    'median weighted Jaccard': [ok.weighted_jaccard.median(), prim.Jw.median()],
    'median Spearman': [ok.spearman.median(), prim.spearman.median()],
    'median Dice top-15%': [ok.dice_top15.median(), prim.dice15.median()]})
base_cmp.to_csv(TAB / 'module1_baseline_comparison.csv', index=False)
display(base_cmp.round(3))
F.fig_m1_baseline(mm, FIG); show('fig_module1_minmax_baseline', 600)

In [ ]:
proj = pd.read_csv(ARCH / 'module1/projection_summary.csv')
print('Sensitivity of the min–max baseline to the corridor half-width (px):')
proj.round(4)

### 5. Module 1 branch analysis: min–max agreement and disagreement (1054 recordings)

The Module 1 branch projects the 2D Grad-CAM with a 13-px vertical maximum-filter mask around the trace (±6 px), min–max normalises both profiles over the valid columns,
and summarises each recording by the mean local agreement $\bar A$, the mean local disagreement $\bar D$, Spearman, top-15 % Dice and IoU, and weighted Jaccard.
A profile with range below $10^{-8}$ is degenerate; similarity metrics use the non-degenerate pairs. The per-recording values were computed by the branch scripts
(`src/hpc_pipeline/02_gradcam_and_module1/module1_alignment_branch/`) and are archived in `data/archived/module1/branch/`; this section recomputes every summary and test from them.

In [ ]:
from scipy.stats import kruskal as kw_test, mannwhitneyu
BR = ARCH / 'module1' / 'branch'
m1b = pd.read_csv(BR / 'agreement_metrics_0px.csv')
m1b['degenerate'] = (m1b.degenerate_1d.astype(str) == 'True') | (m1b.degenerate_2d.astype(str) == 'True')
nd = m1b[~m1b.degenerate].copy()
p1r = pd.read_csv(ARCH / 'replication/1d_test_predictions.csv').set_index('cohort_index')
p2r = pd.read_csv(ARCH / 'replication/2d_test_predictions.csv').set_index('cohort_index')
both = (p1r.true_class == p1r.predicted_class) & (p2r.true_class == p2r.predicted_class)
m1b['both_correct'] = m1b.cohort_index.map(both); nd['both_correct'] = nd.cohort_index.map(both)
SIM = ['spearman', 'dice_top15', 'iou_top15', 'weighted_jaccard']
METRICS = ['mean_agreement', 'mean_disagreement'] + SIM
overall = pd.Series({
    'total recordings': len(m1b), 'non-degenerate pairs': len(nd), 'degenerate pairs': int(m1b.degenerate.sum()),
    'both CNNs correct': int(m1b.both_correct.sum()), 'both correct and non-degenerate': int(nd.both_correct.sum()),
    'mean agreement (1054)': m1b.mean_agreement.mean(), 'mean disagreement (1054)': m1b.mean_disagreement.mean(),
    **{f'{k} (1033)': nd[k].mean() for k in SIM}})
overall.to_csv(TAB / 'module1_branch_overall.csv', header=['value'])
overall.round(4).to_frame('value')

In [ ]:
cls = nd.groupby('class_name')[METRICS].mean(); cls.insert(0, 'n', nd.groupby('class_name').size())
cls.to_csv(TAB / 'module1_branch_classwise.csv')
print('Class-wise means (1033 non-degenerate pairs):')
cls.round(4)

In [ ]:
rows, pw = [], []
k, n = 4, len(nd)
for mtr in METRICS:
    groups = [nd[nd.class_name == c][mtr] for c in CLASSES]
    H, p = kw_test(*groups)
    rows.append({'metric': mtr, 'H': H, 'p': p, 'epsilon_squared': (H - k + 1) / (n - k)})
    raw = []
    for i in range(4):
        for j in range(i + 1, 4):
            a, b = groups[i], groups[j]
            U, pr = mannwhitneyu(a, b, alternative='two-sided')
            raw.append({'metric': mtr, 'pair': f'{CLASSES[i]}–{CLASSES[j]}', 'U': U, 'p_raw': pr,
                        'rank_biserial': 2 * U / (len(a) * len(b)) - 1})
    order = np.argsort([r['p_raw'] for r in raw]); m = len(raw); prev = 0
    for rank, idx in enumerate(order):                       # Holm step-down over the six class pairs
        adj = min(1.0, max(prev, (m - rank) * raw[idx]['p_raw'])); raw[idx]['p_holm'] = adj; prev = adj
    pw += raw
kwt = pd.DataFrame(rows); pwt = pd.DataFrame(pw); pwt['significant'] = pwt.p_holm < .05
kwt.to_csv(TAB / 'module1_branch_kruskal.csv', index=False); pwt.to_csv(TAB / 'module1_branch_pairwise.csv', index=False)
display(kwt.assign(p=kwt.p.map('{:.2e}'.format)).round(4))
for mtr in ['mean_agreement', 'mean_disagreement']:
    print(f'{mtr}: Holm-significant pairs ->', ', '.join(pwt[(pwt.metric == mtr) & pwt.significant].pair))

In [ ]:
sub = []
g1, g0 = nd[nd.both_correct], nd[~nd.both_correct]
for mtr in METRICS:
    U, p = mannwhitneyu(g1[mtr], g0[mtr], alternative='two-sided')
    sub.append({'metric': mtr, 'mean both correct': g1[mtr].mean(), 'mean ≥1 wrong': g0[mtr].mean(), 'p': p,
                'rank_biserial': 2 * U / (len(g1) * len(g0)) - 1})
sub = pd.DataFrame(sub); sub.to_csv(TAB / 'module1_branch_correctness.csv', index=False)
print(f'correct in both CNNs: {len(g1)} | misclassified by at least one: {len(g0)}  (exploratory, no multiplicity correction)')
sub.assign(p=sub.p.map('{:.2e}'.format)).round(4)

In [ ]:
import dan_column_figures as C
C.fig_module1_classwise_column(nd, FIG)
show('col_module1_classwise', 420)

#### 5.1 Sensitivity: temporal shift, projection mask and independent alignment validation

In [ ]:
sh = pd.read_csv(BR / 'shift_metrics.csv')
sh_nd = sh[(sh.degenerate_1d.astype(str) == 'False') & (sh.degenerate_2d.astype(str) == 'False')]
shift_summary = pd.concat([sh.groupby('shift_pixels')[['mean_agreement', 'mean_disagreement']].mean(),
                           sh_nd.groupby('shift_pixels')[SIM].mean()], axis=1)
shift_summary.to_csv(TAB / 'module1_branch_shift_sensitivity.csv')
pr = pd.read_csv(BR / 'projection_metrics.csv')
rng42 = np.random.default_rng(42); boot = []
for mask_size, g in pr.groupby('mask_size'):
    gnd = g[(g.degenerate_1d.astype(str) == 'False') & (g.degenerate_2d.astype(str) == 'False')]
    for mtr in METRICS:
        v = (g if mtr in ('mean_agreement', 'mean_disagreement') else gnd)[mtr].to_numpy()
        b = v[rng42.integers(len(v), size=(2000, len(v)))].mean(1)
        boot.append({'mask_size': mask_size, 'metric': mtr, 'mean': v.mean(), 'ci_lo': np.percentile(b, 2.5), 'ci_hi': np.percentile(b, 97.5)})
proj_boot = pd.DataFrame(boot); proj_boot.to_csv(TAB / 'module1_branch_projection_sensitivity.csv', index=False)
rngs = pd.DataFrame({'shift −5…+5 px: min': shift_summary.min(), 'shift: max': shift_summary.max(),
                     'mask 6/13/24 px: min': proj_boot.pivot(index='mask_size', columns='metric', values='mean').min(),
                     'mask: max': proj_boot.pivot(index='mask_size', columns='metric', values='mean').max()})
display(rngs.loc[METRICS].round(4))
print('Spearman at shift −5 / 0 / +5 px:', shift_summary.spearman.loc[[-5, 0, 5]].round(4).tolist())
C.fig_module1_sensitivity_column(shift_summary, proj_boot, FIG)
show('col_module1_sensitivity', 420)

In [ ]:
tv = pd.read_csv(BR / 'temporal_validation_100.csv')
tvs = tv[['zero_shift_correlation', 'best_correlation', 'correlation_improvement']].agg(['mean', 'median', 'std']).T
tvs.to_csv(TAB / 'module1_branch_temporal_validation.csv')
print(f'recordings: {len(tv)} | selected shift: ' + ', '.join(f'{s:+d} px ({n} ECGs)' for s, n in tv.best_shift_pixels.value_counts().items())
      + f' | {tv.best_shift_ms.mean():.2f} ms')
tvs.round(4)

**Summary of Module 1.** Cross-representation agreement is weak and only marginally above the mispairing null, the 2D explanation is broad,
and disagreement is directional rather than random. This is the evidence that a fused map cannot be read as "both models looked here" without testing.

# Part 3 · Module 2: prediction-preserving, branch-specific stability

**Research question 2.** How stable is each branch's explanation under small changes of the raw ECG that do not change its prediction?

**Perturbation set** (30 per recording, applied to the raw signal; both inputs are regenerated, the image by re-rendering):
amplitude scaling ×{0.8, 0.9, 0.95, 1.05, 1.1, 1.2}; baseline drift at 0.3 Hz, amplitude 0.05 or 0.10, six phases; Gaussian noise σ = 0.05 or 0.10, six seeds.

**Acceptance.** A perturbation contributes to branch $b$ only if branch $b$ keeps its prediction. Prediction retention (PRR) is reported separately.

**Stability.** Over accepted perturbations, $\mu_b(u)$ is the mean attribution and $S_b(u)=1-\sigma_b(u)/(\max_u\sigma_b+\varepsilon)$ the local stability;
$H_b=\mu_b S_b$ is the stability-weighted map used by Module 3.
The maps were produced on Kaggle by `src/hpc_pipeline/03_module2_stability/run_dan_module2_noise_stability.py`; this notebook analyses the archived output.

In [ ]:
from scipy.stats import wilcoxon
rng = np.random.default_rng(20261117)
z = np.load(ARCH / 'module2/noise_stability_maps.npz')
rec = pd.read_csv(ARCH / 'module2/noise_stability_records.csv')
V = z['valid_600'].astype(bool)
rec['both_correct'] = (rec.pred_1d == rec.class_name) & (rec.pred_2d == rec.class_name)
print('perturbations:', list(z['pert_names']))
print('records:', len(rec), '| both correct:', int(rec.both_correct.sum()))

### 1. Prediction retention

In [ ]:
fam = ['scale', 'drift_0.05', 'drift_0.1', 'noise_0.05', 'noise_0.1']
prr = pd.DataFrame({f: {'1D': rec[f'prr_1d_{f}'].mean(), '2D': rec[f'prr_2d_{f}'].mean()} for f in fam}).T
prr['all 30'] = [np.nan] * len(prr)
prr.loc['all 30'] = [rec.prr_1d.mean(), rec.prr_2d.mean(), np.nan]
prr = prr.drop(columns='all 30')
prr.to_csv(TAB / 'module2_prr_by_perturbation.csv')
display((100 * prr).round(2).rename(columns=lambda c: c + ' PRR %'))
acc = rec[rec.both_correct].n_joint_accepted
print(f'jointly accepted perturbations (primary cohort): median {acc.median():.0f}, '
      f'all 30 accepted in {int((acc == 30).sum())} of {len(acc)} recordings ({100 * (acc == 30).mean():.1f} %), minimum {acc.min()}')
print(f'recordings (of {len(rec)}) with fewer than ten jointly accepted perturbations: {int((rec.n_joint_accepted < 10).sum())}')
print(f'degenerate perturbed maps: 1D {int(rec.n_degenerate_1d.sum())}, 2D {int(rec.n_degenerate_2d.sum())}; '
      f'Grad-CAM maps per branch (31 inputs x {len(rec)} ECGs): {31 * len(rec)}')

In [ ]:
acc_cls = rec.groupby('class_name').agg(n=('cohort_index', 'size'),
                                     acc_1d=('pred_1d', lambda x: (x == rec.loc[x.index, 'class_name']).mean() * 100),
                                     acc_2d_rerendered=('pred_2d', lambda x: (x == rec.loc[x.index, 'class_name']).mean() * 100),
                                     both_correct=('both_correct', 'sum'), PRR_1d=('prr_1d', 'mean'), PRR_2d=('prr_2d', 'mean'))
acc_cls.loc['All'] = [len(rec), (rec.pred_1d == rec.class_name).mean() * 100, (rec.pred_2d == rec.class_name).mean() * 100,
                      rec.both_correct.sum(), rec.prr_1d.mean(), rec.prr_2d.mean()]
acc_cls.to_csv(TAB / 'module2_inputs_by_class.csv'); acc_cls.round(3)

### 2. Explanation stability

In [ ]:
EPS = 1e-12
def topk(x, f=.15):
    k = int(np.ceil(f * x.size)); s = np.zeros(x.size, bool); s[np.argsort(-x, kind='stable')[:k]] = True; return s
rows = []
for i in np.flatnonzero(rec.both_correct):
    v = V[i]; mu1, mu2, S1, S2 = (z[k][i][v] for k in ('mu1', 'mu2', 'S1', 'S2'))
    s1, s2 = topk(mu1), topk(mu2)
    rows.append({'cohort_index': rec.cohort_index[i], 'class_name': rec.class_name[i],
                 'S1_mean': S1.mean(), 'S2_mean': S2.mean(),
                 'S1_salient': S1[s1].mean(), 'S1_rest': S1[~s1].mean(),
                 'S2_salient': S2[s2].mean(), 'S2_rest': S2[~s2].mean()})
st = pd.DataFrame(rows); st.to_csv(TAB / 'module2_stability_per_record.csv', index=False)
def ci(x):
    b = [np.median(rng.choice(x, x.size)) for _ in range(2000)]; return '%.3f–%.3f' % tuple(np.percentile(b, [2.5, 97.5]))
tab = st.groupby('class_name')[['S1_mean', 'S2_mean', 'S1_salient', 'S1_rest', 'S2_salient', 'S2_rest']].median()
tab.loc['All'] = st[tab.columns].median()
tab['S2 − S1 (median)'] = list(st.groupby('class_name').apply(lambda g: (g.S2_mean - g.S1_mean).median())) + [(st.S2_mean - st.S1_mean).median()]
tab.to_csv(TAB / 'module2_stability_by_class.csv')
print('95% CI of the median S2 − S1 (all):', ci((st.S2_mean - st.S1_mean).to_numpy()))
print('Wilcoxon S1 vs S2: p = %.2e' % wilcoxon(st.S1_mean, st.S2_mean).pvalue)
print('Wilcoxon 2D salient vs rest: p = %.2e' % wilcoxon(st.S2_salient, st.S2_rest).pvalue)
print('Wilcoxon 1D salient vs rest: p = %.2e' % wilcoxon(st.S1_salient, st.S1_rest).pvalue)
tab.round(3)

In [ ]:
from scipy.stats import pearsonr
r12 = pearsonr(rec.S1_mean, rec.S2_mean)
print(f'correlation of per-recording mean stability S1 vs S2 (all {len(rec)} recordings): r = {r12.statistic:.2f}')
n_first = int(round(V[0].sum() / 10))                 # first second = first 10 % of the valid columns
S2p = np.stack([z['S2'][i][V[i]] for i in np.flatnonzero(rec.both_correct)])
print(f'S2 in the first second: {S2p[:, :n_first].mean():.2f} vs the remaining nine seconds: {S2p[:, n_first:].mean():.2f}')

**Reading.** Both CNNs keep their prediction under most perturbations, but the 2D explanation is much less stable than the 1D explanation,
and in the 2D branch the most attributed columns are less stable than the rest. Prediction stability therefore does not imply explanation stability.

In [ ]:
import dan_figures as F
per = pd.read_csv(TAB / 'module1_per_record.csv')
per = per.merge(st.drop(columns='class_name'), on='cohort_index', how='left')
per['both_correct'] = per.both_correct.fillna(False).astype(bool)
F.fig_module2_detail(z, rec, per, FIG)
show('fig_module2_detail', 800)

Single-column version of the same figure, used in the report:

In [ ]:
import dan_column_figures as C
C.fig_module2_column(z, rec, per, FIG)
show('col_module2', 420)

# Part 4 · Module 3: reliability-weighted fusion (3A) and waveform-conditioned fusion (3B)

**Module 3A.** Stability-weighted maps from Module 2 are fused with a geometric mean, computed over perturbations accepted by **both** branches:
$$R(u)=\sqrt{H_{1D}(u)\,H_{2D}(u)},\qquad H_b=\mu_b S_b .$$
Two fused baselines use the same geometric mean without stability: the anchor-style fusion of the raw Grad-CAM maps, $F_{\mathrm{base}}=\sqrt{B_1B_2}$,
and the fusion of perturbation means, $F_{\mu}=\sqrt{\mu_1\mu_2}$ (defined only for the 551 recordings where all 30 perturbations were accepted by both branches).

**Module 3B.** $R$ is additionally weighted by how much of the 2D Grad-CAM lies on the rendered waveform,
$$F_{\mathrm{trace}}(u)=\frac{\sum_y H(y,u)M(y,u)}{\sum_y H(y,u)+\varepsilon},\qquad R_{\mathrm{trace}}(u)=R(u)\,F_{\mathrm{trace}}(u)^{\lambda},\ \lambda\in\{0.5,1,2\},$$
with $M$ the trace mask dilated by ±6 px. Since $0\le F_{\mathrm{trace}}\le1$, $R_{\mathrm{trace}}\le R$ by construction.

The maps were produced on the HPC (`src/hpc_pipeline/04_module3_fusion_faithfulness/`: `m3a_prepare.py`, `m3b_generate_trace_support.py`, `m3b_apply.py`).
This notebook audits them, recomputes the fusion formulas and summarises the cohort. Faithfulness is evaluated in notebook 05.

### 1. Module 3A eligibility and formula check

In [ ]:
s3a = json.loads((ARCH / 'module3a/module3a_summary.json').read_text())
el = pd.read_csv(ARCH / 'module3a/module3a_eligible_records.csv')
z3 = np.load(ARCH / 'module3a/module3a_maps.npz')
flow = pd.DataFrame({'step': ['Module 2 records', 'frozen Module 1 primary recordings',
                              'eligible for Module 3A (both correct in Module 2, >= 2 joint acceptances, masks and labels match)',
                              'all 30 perturbations jointly accepted (matched-30, used for F_mu)'],
                     'n': [s3a['n_module2'], s3a['n_module1_frozen_primary'], s3a['n_primary_eligible_m2'], s3a['n_primary_matched30_clean']]})
display(flow)
V = z3['valid_600'].astype(bool)
err_R = max(np.abs(z3['R'][i][V[i]] - np.sqrt(np.clip(z3['H1j'][i][V[i]], 0, None) * np.clip(z3['H2j'][i][V[i]], 0, None))).max()
            for i in range(len(V)))
err_F = max(np.abs(z3['Fbase'][i][V[i]] - np.sqrt(np.clip(z3['base1'][i][V[i]], 0, None) * np.clip(z3['base2'][i][V[i]], 0, None))).max()
            for i in range(len(V)))
print(f'max |R - sqrt(H1 H2)| recomputed here: {err_R:.2e}   (HPC audit: {s3a["max_R_formula_error"]:.2e})')
print(f'max |F_base - sqrt(B1 B2)| recomputed here: {err_F:.2e}')
print('median mean R over eligible recordings:', round(s3a['median_r_mean_eligible'], 4),
      '| median intensity ratio R / F_mu (matched-30):', round(s3a['median_matched30_ratio_r_over_mu'], 4))

In [ ]:
by_cls = el.groupby('class_name').agg(n=('cohort_index', 'size'), matched30=('full_joint_30', 'sum'),
                                     median_R_mean=('r_mean', 'median'), median_R_over_Fmu=('r_fmu_intensity_ratio', 'median'),
                                     median_shape_r_R_vs_Fmu=('r_fmu_shape_pearson', 'median'))
by_cls.loc['All'] = [len(el), el.full_joint_30.sum(), el.r_mean.median(), el.r_fmu_intensity_ratio.median(), el.r_fmu_shape_pearson.median()]
by_cls.to_csv(TAB / 'module3A_by_class.csv'); by_cls.round(3)

### 2. What stability weighting changes (694 eligible recordings)

In [ ]:
def topk(x, f=.10):
    k = int(np.ceil(f * x.size)); s = np.zeros(x.size, bool); s[np.argsort(-x, kind='stable')[:k]] = True; return s
def iou(a, b): return (a & b).sum() / (a | b).sum()
def ent(x):
    p = np.clip(x, 0, None); p = p / p.sum(); nz = p[p > 0]; return float(-(nz * np.log(nz)).sum() / np.log(p.size))
z2 = np.load(ARCH / 'module2/noise_stability_maps.npz'); rec = pd.read_csv(ARCH / 'module2/noise_stability_records.csv')
pos = {c: i for i, c in enumerate(z2['cohort_index'])}
rows = []
for cid, cl in zip(el.cohort_index, el.class_name):
    i = pos[cid]; v = z2['valid_600'][i].astype(bool)
    H1, H2, R, mu1, mu2, S1, S2 = (z2[k][i][v] for k in ('H1', 'H2', 'R', 'mu1', 'mu2', 'S1', 'S2'))
    R0 = np.sqrt(np.clip(mu1, 0, None) * np.clip(mu2, 0, None))          # fusion without stability
    rows.append({'cohort_index': cid, 'class_name': cl, 'S1 mean': S1.mean(), 'S2 mean': S2.mean(),
                 'sum H1 / sum mu1': H1.sum() / mu1.sum(), 'sum H2 / sum mu2': H2.sum() / mu2.sum(),
                 'sum R / sum R0': R.sum() / R0.sum(), 'IoU10(R0, R)': iou(topk(R0), topk(R)),
                 'IoU10(H1D, R)': iou(topk(H1), topk(R)), 'IoU10(H2D, R)': iou(topk(H2), topk(R)),
                 'entropy H1D': ent(H1), 'entropy H2D': ent(H2), 'entropy R': ent(R)})
loc = pd.DataFrame(rows); loc.to_csv(TAB / 'module3A_location_per_record.csv', index=False)
m3t = loc.drop(columns='cohort_index').groupby('class_name').median().T[CLASSES]
m3t['All'] = loc.drop(columns=['cohort_index', 'class_name']).median()
m3t.loc['n'] = list(loc.class_name.value_counts()[CLASSES]) + [len(loc)]
m3t = m3t.loc[['n'] + [r for r in m3t.index if r != 'n']]
m3t.to_csv(TAB / 'module3A_table_by_class.csv'); print('Medians over recordings (R0 = fusion without stability, sqrt(mu1 mu2)):')
m3t.round(3)

**Reading.** The top-10 % columns of $R$ coincide much more with those of $H_{1D}$ than with those of $H_{2D}$: the broad, flat 2D map mainly scales the 1D map,
and removes 1D peaks where the 2D map is weak or unstable.

In [ ]:
import dan_figures as F
mask = np.load(ARCH / 'module1/ecg209_trace_mask.npy')
i209 = int(np.flatnonzero(z2['cohort_index'] == 209)[0])
import dan_column_figures as C
F.fig_case_module3a(z2, i209, mask, FIG)
C.fig_module3a_column(z2, i209, mask, FIG)        # single-column version used in the report
show('col_module3A', 420)

In [ ]:
v = z2['valid_600'][i209].astype(bool)
mu1, mu2, S1, S2, R = (z2[k][i209][v] for k in ('mu1', 'mu2', 'S1', 'S2', 'R'))
R0 = np.sqrt(mu1 * mu2)
print(f'ECG 209: mean S1 {S1.mean():.2f}, mean S2 {S2.mean():.2f}; R keeps {100 * R.sum() / R0.sum():.0f} % of the mass of '
      f'fusion without stability; top-10 % IoU(R, R0) = {iou(topk(R), topk(R0)):.2f}')

### 3. Module 3B: waveform conditioning

In [ ]:
z3b = np.load(ARCH / 'module3b/module3b_maps.npz')
m3b = pd.read_csv(ARCH / 'module3b/module3b_record_metrics.csv')
V3 = z3b['valid_600'].astype(bool)
err = 0.0
for lam, key in ((0.5, 'Rtrace_lambda_0p5'), (1.0, 'Rtrace_lambda_1p0'), (2.0, 'Rtrace_lambda_2p0')):
    for i in range(len(V3)):
        v_ = V3[i]; err = max(err, np.abs(z3b[key][i][v_] - z3b['R'][i][v_] * np.clip(z3b['F_trace'][i][v_], 0, None) ** lam).max())
print('records:', len(V3), '| max |R_trace - R F_trace^lambda| recomputed here: %.2e' % err)
print('R_trace <= R everywhere:', all((z3b[k][V3] <= z3b['R'][V3] + 1e-7).all() for k in ('Rtrace_lambda_0p5', 'Rtrace_lambda_1p0', 'Rtrace_lambda_2p0')))
def iqr(s): return '%.3f (%.3f–%.3f)' % (s.median(), s.quantile(.25), s.quantile(.75))
coh = m3b.groupby('lambda').agg(n=('cohort_index', 'size'), mass_kept_vs_R=('Rtrace_over_R_mean', iqr),
                                top10_IoU_vs_R=('top10_IoU', iqr), shape_Pearson_vs_R=('shape_Pearson', iqr))
sens = pd.read_csv(ARCH / 'module3b_sensitivity/module3b_record_metrics.csv')
coh['mass kept, sensitivity run (694, median)'] = sens.groupby('lambda').Rtrace_over_R_mean.median().round(4)
coh.to_csv(TAB / 'module3B_cohort.csv')
on_trace = m3b.drop_duplicates('cohort_index').F_trace_mean
print('median on-trace share of the 2D Grad-CAM per recording: %.3f (IQR %.3f–%.3f)' % (on_trace.median(), on_trace.quantile(.25), on_trace.quantile(.75)))
coh

ECG 1346 failed the strict reproduction check of the 2D map in Module 3B and is excluded from the primary run (693 recordings);
the sensitivity run that keeps it (694) gives the same medians to three decimals.

### 4. Geometric audit of $F_{\mathrm{trace}}$ on ECG 209

A steep QRS complex covers more rows of a pixel column than a flat baseline, so the corridor around the trace is taller there and can capture more 2D mass for purely geometric reasons.
We measure the corridor height of every valid column from the rendered waveform and check whether $F_{\mathrm{trace}}$ and the top-ranked columns of $R_{\mathrm{trace}}$ follow it.

In [ ]:
from scipy.stats import spearmanr
j = int(np.flatnonzero(z3b['cohort_index'] == 209)[0]); v_ = V3[j]
Ft, R209 = z3b['F_trace'][j][v_], z3b['R'][j][v_]
cols = np.array_split(np.arange(mask.shape[1]), v_.sum())       # rendered waveform pixels -> valid time columns
height = np.array([mask[:, c].sum() for c in cols], float)      # trace pixels per time column
tall = topk(height, .10)
geo = {'Spearman(corridor height, F_trace)': spearmanr(height, Ft).statistic,
       'mean F_trace, tallest 10% columns': Ft[tall].mean(), 'mean F_trace, other columns': Ft[~tall].mean()}
for name, key in (('R', None), ('R_trace, lambda=0.5', 'Rtrace_lambda_0p5'), ('R_trace, lambda=1', 'Rtrace_lambda_1p0'), ('R_trace, lambda=2', 'Rtrace_lambda_2p0')):
    x = R209 if key is None else z3b[key][j][v_]
    geo[f'share of top-10% columns of {name} in tallest 10%'] = (topk(x) & tall).sum() / topk(x).sum()
geo = pd.Series(geo).round(3); geo.to_csv(TAB / 'module3B_geometry_check_ecg209.csv', header=['value']); geo

**Reading.** As $\lambda$ grows, the top-ranked columns of $R_{\mathrm{trace}}$ move into the tallest (QRS) columns, where $F_{\mathrm{trace}}$ is high for geometric reasons.
Any faithfulness gain of Module 3B (notebook 05) must therefore be read together with this bias.

In [ ]:
F.fig_module3b(ARCH / 'module3b/module3b_maps.npz', m3b, mask, FIG)
C.fig_module3b_column(ARCH / 'module3b/module3b_maps.npz', m3b, mask, FIG)   # single-column version used in the report
show('col_module3B', 420)

# Part 5 · Faithfulness and comparative evaluation

**Research question 3.** Is the reliability-weighted fused map more faithful to the CNNs than fused maps without stability, than random deletion,
and than its own components? Does waveform conditioning (Module 3B) help?

**Paired block deletion.** Each map is resampled to the 5000 raw samples. For deletion fraction $f\in\{0.05,\dots,0.30\}$,
$n_f=\mathrm{round}(5000f/50)$ non-overlapping 50-sample blocks are chosen greedily by mean attribution and replaced by linear interpolation.
The 1D CNN receives the modified signal and the 2D CNN a re-rendered image. With $d_m(f)$ the drop in true-class probability,
$$\mathrm{AUC}_m=\int_0^{0.30} d_m(f)\,df\ \text{(trapezoid, anchored at }(0,0)),\qquad \Delta\mathrm{AUC}_m=\mathrm{AUC}_m-\mathrm{AUC}_R ,$$
compared per recording (paired), with a 2000-resample bootstrap over recordings. Random control: same number of random blocks, five replicates.

**Data.** 693 Module 3B recordings were submitted; 692 passed the strict check that the archived baseline probability is reproduced (tolerance $10^{-5}$); ECG 406 failed and is not imputed.
The deletion runs were executed on the HPC (`module3_faithfulness.py`, one SLURM array task per recording); this notebook re-checks every raw file and recomputes all statistics.

### 1. Quality control of the 692 raw result files

In [ ]:
RAW = ARCH / 'faithfulness_cohort/raw'
FRACTIONS = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
CORE = {'R', 'Fbase', 'Rtrace_0p5', 'Rtrace_1p0', 'Rtrace_2p0'}
expected = set(range(693)) - {602}                    # index 602 = ECG 406, failed baseline QC
csvs = {int(p.stem.split('_')[1]): p for p in RAW.glob('faith_????.csv')}
audits = {int(p.stem.split('_')[1]): p for p in RAW.glob('faith_????_audit.json')}
assert set(csvs) == expected and set(audits) == expected
m3_ids = np.load(ARCH / 'module3b/module3b_maps.npz')['cohort_index'].astype(int)
frames, aud = [], []
for k in sorted(expected):
    meta = json.loads(audits[k].read_text()); df = pd.read_csv(csvs[k])
    assert meta['strict_baseline_qc'] is True and meta['oneDNN_disabled'] is True
    assert int(meta['cohort_index']) == m3_ids[k] and set(df.cohort_index) == {m3_ids[k]}
    assert set(df.block_samples) == {50} and np.allclose(df.actual_fraction, df.fraction)
    assert np.isfinite(df[['probability_drop', 'baseline_probability', 'perturbed_probability']].to_numpy(float)).all()
    for b in ('1d', '2d'):
        r = df[(df.branch == b) & (df.control == 'random')]
        assert len(r) == 30 and CORE <= set(df[(df.branch == b) & (df.control == 'attribution')].method)
    frames.append(df.assign(index=k)); aud.append(meta)
raw = pd.concat(frames, ignore_index=True); aud = pd.DataFrame(aud)
print('recordings:', raw.cohort_index.nunique(), '| rows:', len(raw), '| matched-30:', int(aud.matched30.sum()))
print('max baseline reproduction error 1D: %.1e, 2D: %.1e' % (aud.baseline_1d_max_error.max(), aud.baseline_2d_max_error.max()))
print('TensorFlow on the HPC:', aud.tensorflow_version.unique(), '| class counts:', raw.drop_duplicates('cohort_index').class_name.value_counts().to_dict())

### 2. Deletion AUC per recording and paired comparison with $R$

In [ ]:
integrate = getattr(np, 'trapezoid', None) or np.trapz
def auc(g):
    g = g.groupby('fraction').probability_drop.mean().sort_index()
    assert np.allclose(g.index, FRACTIONS)
    return float(integrate(np.r_[0., g.to_numpy()], np.r_[0., FRACTIONS]))
A = (raw.groupby(['index', 'cohort_index', 'class_name', 'branch', 'method']).apply(auc)
        .rename('auc_drop').reset_index())
A.to_csv(TAB / 'faithfulness_auc_per_record.csv', index=False)

def bootstrap_mean(v, seed):
    v = np.asarray(v, float); rng = np.random.default_rng(seed)
    b = v[rng.integers(len(v), size=(2000, len(v)))].mean(1)
    return v.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

rows = []
for b in ('1d', '2d'):
    for cl in ['ALL'] + CLASSES:
        sub = A[A.branch == b] if cl == 'ALL' else A[(A.branch == b) & (A.class_name == cl)]
        pt = sub.pivot(index='cohort_index', columns='method', values='auc_drop')
        for m in ['Fbase', 'Fmu_matched30', 'RANDOM', 'Rtrace_0p5', 'Rtrace_1p0', 'Rtrace_2p0']:
            mt = pt[['R', m]].dropna(); d = mt[m] - mt['R']
            seed = 20261115 + len(d) + len(m) + len(cl) + len(b)          # same seeds as the HPC summary script
            mean, lo, hi = bootstrap_mean(d, seed)
            rows.append({'branch': b, 'class_name': cl, 'comparison': f'{m} - R', 'n_ecg': len(d),
                         'mean_delta_auc': mean, 'ci95_lower': lo, 'ci95_upper': hi})
paired = pd.DataFrame(rows); paired.to_csv(TAB / 'faithfulness_paired_delta_auc.csv', index=False)
hpc = pd.read_csv(ROOT / 'data/archived/faithfulness_cohort/hpc_paired_auc_summary.csv')
chk = paired.merge(hpc, on=['branch', 'class_name', 'comparison'], suffixes=('', '_hpc'))
print('rows compared with the HPC summary:', len(chk), '| max |difference| of mean ΔAUC: %.1e, of CI bounds: %.1e' % (
      (chk.mean_delta_auc - chk.mean_delta_auc_hpc).abs().max(),
      max((chk.ci95_lower - chk.ci95_lower_hpc).abs().max(), (chk.ci95_upper - chk.ci95_upper_hpc).abs().max())))
main = paired[paired.class_name == 'ALL'].copy()
main['significant'] = np.where(main.ci95_lower > 0, 'method > R', np.where(main.ci95_upper < 0, 'method < R', 'n.s.'))
main.round(4)

In [ ]:
absA = (A.groupby(['branch', 'method']).auc_drop.agg(['count', 'mean'])
          .rename(columns={'count': 'n', 'mean': 'mean AUC (prob. drop × fraction)'}))
absA.to_csv(TAB / 'faithfulness_auc_by_method.csv'); absA.round(4)

**Reading (692 recordings).**
* **Against other fused maps**, stability weighting helps the 2D CNN: $R$ is more faithful than $F_{\mathrm{base}}$ and $F_{\mu}$ (ΔAUC < 0 with CI below 0); for the 1D CNN the three fusions are equivalent.
* **Against random deletion**, $R$ is better for the 1D CNN but not for the 2D CNN.
* **Module 3B** raises AUC monotonically with $\lambda$ in both CNNs, but notebook 04 shows that part of this comes from moving attribution to steep QRS columns, so the gain is reported as exploratory.

In [ ]:
import dan_figures as F
curves_att = raw[raw.control == 'attribution'].groupby(['branch', 'method', 'fraction']).probability_drop.mean().reset_index()
rnd = raw[raw.control == 'random'].groupby(['cohort_index', 'branch', 'fraction']).probability_drop.mean().reset_index()
curves = pd.concat([curves_att, rnd.groupby(['branch', 'fraction']).probability_drop.mean().reset_index().assign(method='RANDOM')])
curves.to_csv(TAB / 'faithfulness_deletion_curves.csv', index=False)
F.fig_faith692(curves, paired, FIG)
import dan_column_figures as C
C.fig_faith692_column(curves, paired, FIG)        # single-column version used in the report
show('col_module3_faith692', 420)

### 3. Component analysis: is the fused map as faithful as its components? (100 recordings, 25 per class)

The earlier deletion run (same protocol, AOPC-style mean drop over fractions minus matched random deletion) compares, per branch, the raw Grad-CAM,
the stability-weighted map $H=\mu S$ and the fused map $R$.

In [ ]:
rng = np.random.default_rng(20261117)
cr = pd.read_csv(ARCH / 'faithfulness_component/deletion_raw_results.csv')
agg = cr.groupby(['cohort_index', 'class_name', 'branch', 'method', 'control']).probability_drop.mean().unstack('control').reset_index()
agg['gain'] = agg.attribution - agg.random
def boot(x):
    b = [rng.choice(x, x.size).mean() for _ in range(2000)]; return np.percentile(b, [2.5, 97.5])
rows = []
for (b, m), g in agg.groupby(['branch', 'method']):
    lo, hi = boot(g.gain.to_numpy())
    rows.append({'branch': b, 'method': m, 'n': len(g), 'gain over random (mean)': g.gain.mean(), 'CI lo': lo, 'CI hi': hi,
                 '% records gain > 0': 100 * (g.gain > 0).mean()})
comp = pd.DataFrame(rows); comp.to_csv(TAB / 'component_gain_over_random.csv', index=False)
pairs = []
for b, H, base in (('1d', 'H1', 'base1'), ('2d', 'H2', 'base2')):
    g = agg[agg.branch == b].pivot(index='cohort_index', columns='method', values='attribution')
    for a_, b_ in ((H, base), ('R', H), ('R', base)):
        d = (g[a_] - g[b_]).dropna().to_numpy(); lo, hi = boot(d)
        pairs.append({'branch': b, 'comparison': f'{a_} - {b_}', 'n': len(d), 'mean diff': d.mean(), 'CI lo': lo, 'CI hi': hi})
pairs = pd.DataFrame(pairs); pairs.to_csv(TAB / 'component_paired.csv', index=False)
display(comp.round(4)); display(pairs.round(4))
from scipy.stats import wilcoxon, spearmanr
for b, H, base in (('1d', 'H1', 'base1'), ('2d', 'H2', 'base2')):
    g = agg[agg.branch == b].pivot(index='cohort_index', columns='method', values='attribution')
    print(b.upper(), ' | '.join(f'{x} vs {y}: Wilcoxon p = {wilcoxon(g[x], g[y]).pvalue:.1e}' for x, y in ((H, base), ('R', H), ('R', base))))

In [ ]:
m1o = pd.read_csv(TAB / 'module1_per_record.csv').set_index('cohort_index').O
s2m = pd.read_csv(ARCH / 'module2/noise_stability_records.csv').set_index('cohort_index').S2_mean
win = {}
for b, H, base in (('1d', 'H1', 'base1'), ('2d', 'H2', 'base2')):
    g = agg[agg.branch == b].pivot(index='cohort_index', columns='method', values='attribution')
    pen = g['R'] - g[H]
    ro, po = spearmanr(pen, m1o.reindex(pen.index)); rs, ps = spearmanr(pen, s2m.reindex(pen.index))
    win[b] = 100 * ((g['R'] > g[H]) & (g['R'] > g[base])).mean()
    print(f'{b.upper()}: fusion penalty R - {H} vs Module 1 shared mass O: rho = {ro:.2f} (p = {po:.2f}); '
          f'vs mean S2: rho = {rs:.2f} (p = {ps:.3f}); R above both branch maps in {win[b]:.0f} % of recordings')

In [ ]:
bycls = agg.groupby(['branch', 'method', 'class_name']).gain.mean().unstack('class_name')[CLASSES]
bycls.to_csv(TAB / 'component_gain_by_class.csv')
print('Gain over random deletion by class (100 recordings, 25 per class):')
bycls.round(3)

In [ ]:
summ = pd.read_csv(ARCH / 'faithfulness_component/deletion_summary_bootstrap.csv')
F.fig_faith(summ, pairs.rename(columns={'mean diff': 'mean_diff', 'CI lo': 'ci_lo', 'CI hi': 'ci_hi'}), FIG)
show('fig_component_faithfulness', 750)

### 4. Comparative evaluation summary

In [ ]:
def verdict(b, comp_):
    r = main[(main.branch == b) & (main.comparison == comp_)].iloc[0]
    return f"{r.mean_delta_auc:+.3f} [{r.ci95_lower:+.3f}, {r.ci95_upper:+.3f}]"
summary = pd.DataFrame({
    'comparison (method − R, ΔAUC, 95% CI)': ['anchor-style fusion F_base', 'fusion without stability F_mu (551)', 'random deletion',
                                              'R_trace λ=0.5', 'R_trace λ=1', 'R_trace λ=2'],
    '1D CNN': [verdict('1d', c) for c in ['Fbase - R', 'Fmu_matched30 - R', 'RANDOM - R', 'Rtrace_0p5 - R', 'Rtrace_1p0 - R', 'Rtrace_2p0 - R']],
    '2D CNN': [verdict('2d', c) for c in ['Fbase - R', 'Fmu_matched30 - R', 'RANDOM - R', 'Rtrace_0p5 - R', 'Rtrace_1p0 - R', 'Rtrace_2p0 - R']]})
summary.to_csv(TAB / 'comparative_evaluation_summary.csv', index=False); summary

Negative values mean the method is **less** faithful than $R$. The fused map is a small improvement over fused maps without stability (2D CNN),
it is not better than random deletion for the 2D CNN, and in the component study it is less faithful than each branch's own stability-weighted map.
Fused explanations should therefore be reported together with their components and their disagreement.

# Part 6 · End-to-end pipeline and reproducibility audit

Notebooks 01–05 start from the archived outputs of the HPC runs. This notebook documents how those outputs were produced and checks that the archived files are the ones the HPC wrote.

| Stage | Script(s) in `src/hpc_pipeline/` | Needs | Output used here |
|---|---|---|---|
| 1. Cohort, split, training | `01_data_and_training/build_final_1d.py`, `build_frozen_split.py`, `train_1d_author_code.py`, `train_2d_author_code.py` | raw ECG CSVs, rendered images, GPU | `data/archived/split/`, `replication/` |
| 2. Grad-CAM, alignment, Module 1 | `02_gradcam_and_module1/run_gradcam_*_author_code.py`, `run_dan_module1_*.py`, `module1_alignment_branch/` | trained models | `module1/` |
| 3. Module 2 stability (run on Kaggle) | `03_module2_stability/run_dan_module2_noise_stability.py` | models, ECGs, renderer | `module2/` |
| 4. Module 3A / 3B | `04_module3_fusion_faithfulness/m3a_prepare.py`, `m3b_generate_trace_support.py`, `m3b_apply.py` | Module 2 output, 2D model | `module3a/`, `module3b/`, `module3b_sensitivity/` |
| 5. Deletion faithfulness | `module3_faithfulness.py` (SLURM array 0–692), `summarize_692_exploratory.py` | models, ECGs, renderer | `faithfulness_cohort/` |

Stages 1–5 need the Chapman–Shaoxing data, the trained `.keras` models and TensorFlow. Module 2 ran on Kaggle (TensorFlow 2.21); the other stages ran on the KFUPM HPC (CPU partition, TensorFlow 2.19 with oneDNN disabled for Module 3).
Set `RUN_ON_HPC = True` and run this notebook on the HPC to execute a short smoke test of stages 4–5 for ECG 209.

In [ ]:
RUN_ON_HPC = False                     # set True on the HPC, inside ~/mm_gradcam with the project .venv active
HPC_BASE = Path.home() / 'mm_gradcam' if RUN_ON_HPC else Path('~/mm_gradcam')   # project folder on the HPC
import platform, sklearn, scipy, matplotlib
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__,
      '| scikit-learn', sklearn.__version__, '| matplotlib', matplotlib.__version__)

### 1. Integrity of the archived files against the HPC SHA-256 manifest

In [ ]:
import hashlib
manifest = json.loads((ARCH / 'provenance/hpc_manifest_sha256.json').read_text())
by_name = {}
for k, v in manifest.items():
    by_name.setdefault(Path(k).name, []).append(v['sha256'])
rows = []
for p in sorted(ARCH.rglob('*')):
    if p.is_file() and p.name in by_name and 'provenance' not in p.parts:
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        rows.append({'file': str(p.relative_to(ARCH)), 'match': h in by_name[p.name]})
chk = pd.DataFrame(rows)
print(f'archived files listed in the HPC manifest: {len(chk)} | SHA-256 match: {int(chk.match.sum())} | mismatch: {int((~chk.match).sum())}')
chk.groupby(chk.file.str.split('/').str[0]).match.agg(['count', 'sum']).rename(columns={'count': 'files', 'sum': 'matching'})

In [ ]:
mb = json.loads((ARCH / 'provenance/module1_branch_files.json').read_text())
ok = {k: hashlib.sha256((ARCH / 'module1' / 'branch' / k).read_bytes()).hexdigest() == v['sha256'] for k, v in mb['files'].items()}
print(f"Module 1 branch files (branch {mb['source_branch']}, commit {mb['source_commit']}): {sum(ok.values())} of {len(ok)} match their recorded SHA-256")

### 2. Run configuration recorded by the HPC

In [ ]:
m2 = json.loads((ARCH / 'module2/run_metadata.json').read_text())
cfg = m2['config']
print('Module 2: records', m2['n_records'], '| TensorFlow', m2['tf'], '| scaler fitted on', m2['scaler_n_fit'], 'recordings')
print('  scales', cfg['SCALES'], '| drift', cfg['DRIFT_HZ'], 'Hz', cfg['DRIFTS'], 'x', len(cfg['DRIFT_PHASES']), 'phases',
      '| noise', cfg['NOISES'], 'x', cfg['NOISE_SEEDS'], 'seeds')
print('  waveform columns', cfg['X0'], '-', cfg['X1'], '| grid', cfg['W'], '| corridor ±', cfg['MASK_DILATE_Y'], 'px | min accepted', cfg['MIN_ACCEPTED'])
a0 = json.loads((ARCH / 'faithfulness_cohort/raw/faith_0000_audit.json').read_text())
print('Faithfulness: block', a0['block_samples'], 'samples | fractions', a0['fractions'], '| random repeats', a0['random_repeats'],
      '| seed', a0['seed'], '| TensorFlow', a0['tensorflow_version'], '| oneDNN disabled', a0['oneDNN_disabled'])

### 3. Documented reproduction failures

In [ ]:
for f in sorted((ARCH / 'provenance/diagnostics_ecg406').glob('m3faith_*602.log')):
    hits = [l.strip() for l in f.read_text(errors='ignore').splitlines() if 'QC mismatch' in l or 'baseline' in l.lower()]
    print(f.name, '->', hits[-1] if hits else 'no QC line')
import re
for f in sorted((ARCH / 'provenance/diagnostics_ecg406').glob('m3b_debug406_*.log')):
    txt = f.read_text(errors='ignore')
    mode = re.search(r'MODE=(\w+)', txt).group(1); err = float(re.search(r'DBG406 ERROR: ([0-9.e-]+)', txt).group(1))
    print(f'renderer {mode:5s}: 2D baseline probability error {err:.2e} -> {"PASS" if "PASS" in txt else "FAIL"} (tolerance 1e-5)')
prov = json.loads((ARCH / 'provenance/hpc_provenance.json').read_text())
print({k: prov[k] for k in ['faithfulness_success', 'faithfulness_failed_ecg', 'module3a_eligible', 'module3b_strict', 'module3b_sensitivity',
                             'model_checkpoints_included', 'raw_ecg_included']})

ECG 406 failed the strict 2D baseline check because the matplotlib renderer keeps state between calls: a fresh renderer and a renderer that had drawn other recordings
produced different image bytes. The 2D deletion results are therefore reported as exploratory, and ECG 406 is excluded rather than imputed.

### 4. Smoke test on the HPC (stages 4–5 for one recording)

In [ ]:
import subprocess, shlex
M3 = HPC_BASE / 'module3_aligned'; SRC = HPC_BASE / 'module3_sources'
M3A_DIR = M3 / 'outputs' / 'module3a_v1'          # Module 3A output folder as named on the HPC
steps = [
    f"python src/hpc_pipeline/04_module3_fusion_faithfulness/m3b_generate_trace_support.py --m2-dir {M3}/inputs/module2_s9534 "
    f"--m3a-dir {M3A_DIR} --runtime-root {SRC}/module2_runtime --module2-common src/hpc_pipeline/03_module2_stability/module2_common.py "
    f"--only-ids 209 --out results/smoke/trace_209.npz",
    "python src/hpc_pipeline/04_module3_fusion_faithfulness/module3_faithfulness.py --index 0 --out-dir results/smoke/faith",
]
if RUN_ON_HPC:
    for cmd in steps:
        print('$', cmd); out = subprocess.run(shlex.split(cmd), cwd=ROOT, capture_output=True, text=True)
        print(out.stdout[-2000:], out.stderr[-2000:])
else:
    print('RUN_ON_HPC is False: the commands below are printed, not executed (they need the models, ECGs and TensorFlow).')
    for cmd in steps:
        print('$', cmd)